# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question.** Which content pages should a reviewer look at first because their search visibility is likely to fall over the next 30 days, and does a learned model rank them better than a transparent rule?

**Decision supported.** Each review cycle a content reviewer has time for roughly 20 to 50 pages out of thousands. The ranked queue tells them where to look first and why (reason codes), and which kind of action fits: refresh or improve, protect, merge or prune, or monitor.

**Frame.** For content reviewers deciding which pages to review first, we build a ranked review queue from the FlyRank ML Internship warehouse (Google Search Console and Analytics daily performance plus content metadata). The queue scores the probability that a page's impressions persistently fall, relative to its client, over the next 30 days, and is judged by precision@K against a transparent rule baseline on a sealed test month. A wrong call costs reviewer time (a false alarm) or a longer-undetected decline (a miss). A plain rule is not enough because decline depends on many tangled, shifting signals, and a flag alone cannot order a very large flagged set. We claim only observed, directional, and decision-support results: no causal effect of any refresh, and nothing about how Google ranks pages.

In [8]:
K_LIST = [20, 50]  # review capacities the queue is judged at
print('Judged at K =', K_LIST)

Judged at K = [20, 50]


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release.** The full pseudonymized warehouse release of the FlyRank ML Internship dataset (hosted on Hugging Face, access-gated). No client names, domains, URLs, queries, or raw exports appear in this work.

**Tables used**
- `fact_content_daily_performance`: one row per content item per day; 78,835,655 rows from 2025-01-27 to 2026-06-30. Exact duplicate rows (6,390 groups, 0.016%) are removed with `SELECT DISTINCT`.
- `dim_content`: 519,606 content items (static attributes, creation date, status).
- `dim_clients`: client history start dates, used for eligibility.
- `fact_content_query_90d`: a single window (2026-04-02 to 2026-06-30) that overlaps the label windows, so it is not a model feature.

**Date windows.** For each month-end snapshot t, features come from t-90d to t and the label from t to t+30d. Train: t = 2025-04-30 to 2026-03-31. Validation: t = 2026-04-30. Sealed test: t = 2026-05-31 (label window June 2026). t = 2026-06-30 is used only for the final ranked list.

**Modeling universe.** Published, not-deleted pages at least 90 days old, from clients with at least 90 days of history at t, with at least 100 impressions in the 30 days before t. Pages below the floor are reported as uncovered and not labeled (about 43% of active pages in 2026-03).

**Excluded, and why.** Deleted or unpublished pages (they carried negligible traffic in 2026-03; 411,540 of 519,606 pages are kept). `content_updated_date` (82.5% of dated pages show a May to Jul 2026 update, which looks like a bulk touch). `last_optimized_date` as a feature (no training-period coverage; used only as a flag). Export-time status fields, AI-referral sessions (sparse), and GA4 metrics unless explicitly available. Product flags (health score, priority score, action type) are not in the data and are never used.

Full contract, with the query behind every number: `w03_data_contract.ipynb`.

In [9]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Design constants used by every later section
FLOOR = 100          # min impressions in the 30 days before t
FEATURE_DAYS = 90
LABEL_DAYS = 30
SNAPSHOTS = {
    'train': ['2025-04-30', '2025-05-31', '2025-06-30', '2025-07-31', '2025-08-31', '2025-09-30',
              '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31'],
    'validation': '2026-04-30',
    'test': '2026-05-31',
    'final_list': '2026-06-30',
}

# Cheap checks that the numbers in the text above still hold (full queries: w03_data_contract.ipynb)
con.sql(f"SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(*) AS n_rows FROM {TABLES['fact_daily']}").show()
con.sql(f"SELECT is_published, is_deleted, COUNT(*) AS n FROM {TABLES['dim_content']} GROUP BY 1,2 ORDER BY n DESC").show()
con.sql(f"SELECT MIN(gsc_data_start) AS earliest, MAX(gsc_data_start) AS latest FROM {TABLES['dim_clients']}").show()

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│   min_d    │   max_d    │  n_rows  │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │ 78835655 │
└────────────┴────────────┴──────────┘

┌──────────────┬────────────┬────────┐
│ is_published │ is_deleted │   n    │
│   boolean    │  boolean   │ int64  │
├──────────────┼────────────┼────────┤
│ true         │ false      │ 411540 │
│ false        │ true       │ 101559 │
│ false        │ false      │   6507 │
└──────────────┴────────────┴────────┘

┌────────────┬────────────┐
│  earliest  │   latest   │
│    date    │    date    │
├────────────┼────────────┤
│ 2025-01-27 │ 2026-06-02 │
└────────────┴────────────┘



### Label

**Target `decline_next_30d`**, scored at each month-end snapshot t.

- `last30` = the page's impressions in the 30 days up to and including t. `f1` and `f2` = its impressions in days 1 to 15 and days 16 to 30 after t.
- Page ratio = (f1 + f2) / last30. Client ratio = the same ratio summed over the client's eligible pages (like for like). `rel` = page ratio / client ratio.
- Label = 1 when `rel` is at most 0.5 and both halves are below baseline relative to the client: (f1 / last30) / (client f1 / client last30) < 1, and the same for f2. Otherwise 0.
- Not labeled: pages under 100 impressions in the 30 days before t, and clients with too few eligible pages for a stable benchmark (minimum set below).

**Why relative to the client.** A client-wide shift (seasonality, a tracking change) moves every page together. Comparing each page with its own client keeps those out of the label.

**Why like for like.** Benchmarking against all of a client's pages inflated the benchmark with new, ramping pages. At t = 2026-03-31 the median relative ratio was 0.772 against all pages and 0.837 against eligible pages (64,807 pages).

**Why a 0.5 cut.** Base rates with persistence at cut 0.8 were 25.0% (2026-01-31), 31.4% (2026-02-28) and 44.6% (2026-03-31), so a random queue would already score near the base rate. At cut 0.5 they were 6.7%, 10.2% and 19.9%. The cut was chosen by meaning and base rate on training snapshots, never by model performance, and the sealed test month was not touched.

**The base rate drifts.** The typical page's raw 30-day ratio fell from 1.26 to 1.081 to 0.756 across those three snapshots. Precision@K is therefore not comparable across months, and every result is reported with its base rate and lift (precision divided by base rate).

**Minimum client size.** A client needs at least 100 eligible pages at t to get a label, because a benchmark built from fewer pages is unstable. At t = 2026-03-31 this keeps 16 of 26 clients and 64,669 of 64,807 pages (99.8%).

**Concentration.** At that snapshot six clients with 2,000+ eligible pages hold 89.2% of eligible pages, and the three largest hold 59.3%. Pooled results mostly reflect these clients, so Results reports per-client breakdowns and a client-held-out check, and claims are limited to this client mix.

**Training set.** Twelve month-end snapshots (2025-04-30 to 2026-03-31) give 219,258 labeled rows and 22,429 positives (10.2%), after the 200-page client minimum and the exclusion of new or spiking pages, with up to 13 clients per snapshot. The base rate ranges from 3.2% to 21.4% by snapshot. The first six snapshots (Apr to Sep 2025) have only 1 to 5 clients each (35,090 rows, 16.0%); a sensitivity run without them is reported. The same page appears in many snapshots, so rows are not independent: validation is by time, with a client-held-out check as a robustness test.

### Features

Every feature uses data up to and including t; the label uses data after t only. Pages are the unit; all windows are 30 days (90 for the full feature window).

| Group | Features |
|---|---|
| Volume and engagement (90 days) | `imp_90`, `clicks_90`, `ctr_90`, `ctr_last30`, `active_days_90`, `cv_90` (volatility of daily impressions) |
| Position | `pos_90`, `pos_last30`, `pos_prev30`, `pos_change` (impression-weighted: sum of position / impressions) |
| Momentum | `imp_last30`, `imp_prev30`, `imp_prev60`, `log_mom`, `log_mom_prev`, and the client-relative versions `log_mom_rel`, `log_mom_prev_rel` (client benchmark over the same eligible pages) |
| Age | `page_age_days` at t |
| Static content fields (export-time snapshot) | word and character counts, content type, intent, competition, search volume, cpc, backlinks, category count, keyword tokens, URL length. Reported with and without; `backlinks` may carry information from after t. |

Not used in the main model: GA4 metrics (ablation only), client identity and client size, `last_optimized_date`, `fact_query_90d`.

In [10]:
import pandas as pd

def build_label_base(t_date, benchmark='universe', table='label_base'):
    """One row per eligible page at snapshot t, with the label ingredients.
    benchmark='universe': client ratio computed over the same eligible pages (like for like).
    benchmark='all': client ratio computed over every page of the client (the first version)."""
    t = pd.Timestamp(t_date)
    half = LABEL_DAYS // 2
    months = sorted({(t + pd.Timedelta(days=k)).strftime('%Y-%m') for k in (-(LABEL_DAYS - 1), 0, 1, LABEL_DAYS)})
    months_sql = ', '.join(f"'{m}'" for m in months)
    bench_src = 'elig' if benchmark == 'universe' else 'page'
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id, gsc_impressions
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {LABEL_DAYS}
              AND report_date <= DATE '{t_date}' + {LABEL_DAYS}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, SUM(last30) AS c_last30, SUM(f1) AS c_f1, SUM(f2) AS c_f2
            FROM {bench_src}
            GROUP BY 1
        )
        SELECT e.client_hash_id, e.content_hash_id, e.last30, e.f1, e.f2,
               1.0 * (e.f1 + e.f2) / e.last30 AS raw,
               1.0 * (c.c_f1 + c.c_f2) / c.c_last30 AS c_ratio,
               (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
               (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
               (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2
        FROM elig e
        JOIN cl c ON c.client_hash_id = e.client_hash_id
        WHERE c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
    """)

def centering_summary(table, label):
    return con.sql(f"""
        SELECT '{label}' AS run, COUNT(*) AS n_pages,
               ROUND(quantile_cont(raw, 0.5), 3) AS median_raw_ratio,
               ROUND(quantile_cont(rel, 0.5), 3) AS median_rel,
               ROUND(AVG(CAST(rel <= 0.8 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_8,
               ROUND(AVG(CAST(rel <= 0.5 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_5
        FROM {table}
    """)

# D1. Same snapshot, two benchmarks: all pages of the client vs the eligible pages only
build_label_base('2026-03-31', benchmark='all', table='lb_all')
build_label_base('2026-03-31', benchmark='universe', table='lb_universe')
centering_summary('lb_all', 'Mar-31, benchmark = all pages').show()
centering_summary('lb_universe', 'Mar-31, benchmark = eligible pages').show()

# D2. Placebo: the same like-for-like definition at two other snapshots
for t_other in ['2026-02-28', '2026-01-31']:
    build_label_base(t_other, benchmark='universe', table='lb_other')
    centering_summary('lb_other', f'{t_other}, benchmark = eligible pages').show()

# D3. How big and how fast-moving are the client benchmarks? (client_ratio > 1 = client grew)
con.sql("""
    SELECT COUNT(*) AS n_clients,
           ROUND(quantile_cont(c_ratio, 0.5), 3) AS median_client_ratio,
           quantile_cont(n_pages, [0.1, 0.5, 0.9]) AS pages_per_client_q10_50_90
    FROM (SELECT client_hash_id, ANY_VALUE(c_ratio) AS c_ratio, COUNT(*) AS n_pages
          FROM lb_universe GROUP BY 1)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│              run              │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│            varchar            │  int64  │      double      │   double   │      double       │      double       │
├───────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ Mar-31, benchmark = all pages │   64809 │            0.756 │      0.772 │             0.507 │             0.229 │
└───────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘

┌────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                run                 │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│              varchar               │  int64  │      double 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                  run                   │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│                varchar                 │  int64  │      double      │   double   │      double       │      double       │
├────────────────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ 2026-02-28, benchmark = eligible pages │   55525 │            1.081 │      0.963 │             0.314 │             0.102 │
└────────────────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                  run                   │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│                varchar                 │  int64  │      double      │   double   │      double       │      double       │
├────────────────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ 2026-01-31, benchmark = eligible pages │   41324 │             1.26 │       1.07 │              0.25 │             0.067 │
└────────────────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘

┌───────────┬─────────────────────┬────────────────────────────┐
│ n_clients │ median_client_ratio │ pages_per_client_q10_50_90 │
│   int64   │       double        │          double[]          │
├───────────┼─────────────────────┼───────────────────

In [11]:
# Client size and dominance at 2026-03-31 (like-for-like benchmark table from the previous cell)
con.sql("""
    WITH c AS (SELECT client_hash_id, COUNT(*) AS n_pages FROM lb_universe GROUP BY 1)
    SELECT m AS min_pages_per_client,
           COUNT(*) FILTER (WHERE n_pages >= m) AS clients_kept,
           SUM(n_pages) FILTER (WHERE n_pages >= m) AS pages_kept
    FROM c CROSS JOIN (VALUES (10), (30), (50), (100)) AS t(m)
    GROUP BY m
    ORDER BY m
""").show()

con.sql("""
    WITH c AS (SELECT client_hash_id, COUNT(*) AS n_pages FROM lb_universe GROUP BY 1),
         r AS (SELECT n_pages, ROW_NUMBER() OVER (ORDER BY n_pages DESC) AS rk FROM c)
    SELECT ROUND(1.0 * SUM(n_pages) FILTER (WHERE rk <= 1) / SUM(n_pages), 3) AS top1_client_share,
           ROUND(1.0 * SUM(n_pages) FILTER (WHERE rk <= 3) / SUM(n_pages), 3) AS top3_client_share
    FROM r
""").show()

con.sql("""
    WITH s AS (SELECT *, COUNT(*) OVER (PARTITION BY client_hash_id) AS n_pages FROM lb_universe)
    SELECT CASE WHEN n_pages < 30 THEN 'a: under 30'
                WHEN n_pages < 200 THEN 'b: 30-199'
                WHEN n_pages < 2000 THEN 'c: 200-1999'
                ELSE 'd: 2000+' END AS eligible_pages_in_client,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(*) AS n_pages,
           ROUND(AVG(CAST(rel <= 0.5 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_5
    FROM s
    GROUP BY 1
    ORDER BY 1
""").show()

┌──────────────────────┬──────────────┬────────────┐
│ min_pages_per_client │ clients_kept │ pages_kept │
│        int32         │    int64     │   int128   │
├──────────────────────┼──────────────┼────────────┤
│                   10 │           22 │      64794 │
│                   30 │           18 │      64741 │
│                   50 │           16 │      64669 │
│                  100 │           16 │      64669 │
└──────────────────────┴──────────────┴────────────┘

┌───────────────────┬───────────────────┐
│ top1_client_share │ top3_client_share │
│      double       │      double       │
├───────────────────┼───────────────────┤
│             0.261 │             0.593 │
└───────────────────┴───────────────────┘

┌──────────────────────────┬───────────┬─────────┬───────────────────┐
│ eligible_pages_in_client │ n_clients │ n_pages │ base_rate_cut_0_5 │
│         varchar          │   int64   │  int64  │      double       │
├──────────────────────────┼───────────┼─────────┼──────

In [12]:
REL_MAX = 0.5            # locked: relative ratio at most 0.5
MIN_CLIENT_PAGES = 100   # a client needs this many eligible pages at t for a stable benchmark

def build_labels(t_date, table='labels'):
    """Final label for snapshot t. One row per eligible page; y = decline_next_30d."""
    t = pd.Timestamp(t_date)
    half = LABEL_DAYS // 2
    months = sorted({(t + pd.Timedelta(days=k)).strftime('%Y-%m') for k in (-(LABEL_DAYS - 1), 0, 1, LABEL_DAYS)})
    months_sql = ', '.join(f"'{m}'" for m in months)
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id, gsc_impressions
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {LABEL_DAYS}
              AND report_date <= DATE '{t_date}' + {LABEL_DAYS}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, SUM(last30) AS c_last30, SUM(f1) AS c_f1, SUM(f2) AS c_f2,
                   COUNT(*) AS n_elig
            FROM elig
            GROUP BY 1
        ),
        scored AS (
            SELECT e.client_hash_id, e.content_hash_id, DATE '{t_date}' AS snap_t,
                   e.last30, e.f1, e.f2, c.n_elig,
                   (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
                   (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
                   (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2
            FROM elig e
            JOIN cl c ON c.client_hash_id = e.client_hash_id
            WHERE c.n_elig >= {MIN_CLIENT_PAGES}
              AND c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
        )
        SELECT *, CAST(rel <= {REL_MAX} AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER) AS y
        FROM scored
    """)

# Base rate per training snapshot (the sealed test snapshot is not touched here)
rows = []
for t_str in SNAPSHOTS['train']:
    build_labels(t_str, table='lb_tmp')
    n, k, pos, br = con.sql("""
        SELECT COUNT(*), COUNT(DISTINCT client_hash_id), COALESCE(SUM(y), 0), ROUND(AVG(y), 3)
        FROM lb_tmp
    """).fetchone()
    rows.append((t_str, n, k, pos, br))
label_summary = pd.DataFrame(rows, columns=['snapshot_t', 'n_pages', 'n_clients', 'n_pos', 'base_rate'])
label_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,snapshot_t,n_pages,n_clients,n_pos,base_rate
0,2025-04-30,1482,1,51,0.034
1,2025-05-31,3988,2,632,0.158
2,2025-06-30,7390,4,891,0.121
3,2025-07-31,8172,4,788,0.096
4,2025-08-31,9024,4,2082,0.231
5,2025-09-30,9755,6,2086,0.214
6,2025-10-31,16036,7,968,0.060
7,2025-11-30,21827,8,1203,0.055
8,2025-12-31,33197,11,3162,0.095
9,2026-01-31,41218,14,2760,0.067


In [ ]:
# Is missingness a client fingerprint? One row per client, ranked by size (ids not shown)
con.sql("""
    WITH s AS (
        SELECT client_hash_id, COUNT(*) AS n,
               AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0 END) AS miss_word_count,
               AVG(CASE WHEN backlinks IS NULL THEN 1.0 ELSE 0 END) AS miss_backlinks,
               AVG(CASE WHEN competition IS NULL THEN 1.0 ELSE 0 END) AS miss_keyword_fields,
               AVG(y) AS base_rate
        FROM train_set
        GROUP BY 1
    )
    SELECT ROW_NUMBER() OVER (ORDER BY n DESC) AS client_rank, n,
           ROUND(miss_word_count, 2) AS miss_word_count,
           ROUND(miss_backlinks, 2) AS miss_backlinks,
           ROUND(miss_keyword_fields, 2) AS miss_keyword_fields,
           ROUND(base_rate, 3) AS base_rate
    FROM s
    ORDER BY client_rank
""").show(max_rows=30)

# Does missing mean something for the outcome?
con.sql("""
    SELECT word_count IS NULL AS word_count_missing, backlinks IS NULL AS backlinks_missing,
           COUNT(*) AS n, ROUND(AVG(y), 3) AS base_rate
    FROM train_set
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show()

CatalogException: Catalog Error: Table with name train_set does not exist!
Did you mean "pg_attribute"?

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

K_LIST = [20, 50]
d = con.sql("""
    SELECT snap_t, y, log_mom_rel, log_mom, imp_prev30, imp_last30
    FROM train_set
""").df()

# Rule baselines: a higher score means "look at this page first"
d['rule_rel_momentum'] = -d['log_mom_rel']                      # fell most, relative to its client
d['rule_raw_momentum'] = -d['log_mom']                          # fell most, no client adjustment
d['rule_impressions_lost'] = d['imp_prev30'] - d['imp_last30']  # lost the most impressions in absolute terms
RULES = ['rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

# Summary: mean over the 12 training snapshots. A random queue has lift 1.0 by construction.
summary = []
for r in RULES:
    e = eval_rule(d, r)
    summary.append({'rule': r, **e[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()})
print(pd.DataFrame(summary).to_string(index=False))

# Per-snapshot detail for the main rule
eval_rule(d, 'rule_rel_momentum')

CatalogException: Catalog Error: Table with name train_set does not exist!
Did you mean "pg_attribute"?

In [13]:
import os

assert FEATURE_DAYS == 3 * LABEL_DAYS, 'trend windows assume 90 = 3 x 30'

PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']

def build_snapshot(t_date, table='snap'):
    """Labels (same rules as build_labels) plus past-only features for one snapshot t.
    Features use report_date <= t only; the label uses report_date > t only."""
    t = pd.Timestamp(t_date)
    L = LABEL_DAYS
    half = L // 2
    days = pd.date_range(t - pd.Timedelta(days=FEATURE_DAYS - 1), t + pd.Timedelta(days=L))
    months_sql = ', '.join(f"'{m}'" for m in sorted({d.strftime('%Y-%m') for d in days}))
    static_sql = ', '.join(f"d.{c}" for c in STATIC_COLS)
    static_out = ', '.join(f"e.{c}" for c in STATIC_COLS)
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id,
                   gsc_impressions, gsc_clicks, gsc_sum_position
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {FEATURE_DAYS}
              AND report_date <= DATE '{t_date}' + {L}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   -- label side: strictly after t
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2,
                   -- feature side: up to and including t
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS imp_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_clicks ELSE 0 END) AS clicks_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_sum_position ELSE 0 END) AS pos_sum_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions * gsc_impressions ELSE 0 END) AS imp_sq_90,
                   COUNT(*) FILTER (WHERE report_date <= DATE '{t_date}' AND gsc_impressions > 0) AS active_days_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_clicks ELSE 0 END) AS clicks_last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_sum_position ELSE 0 END) AS pos_sum_last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_impressions ELSE 0 END) AS prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_sum_position ELSE 0 END) AS pos_sum_prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {2*L} THEN gsc_impressions ELSE 0 END) AS prev60
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*, d.content_created_date, {static_sql}
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, COUNT(*) AS n_elig,
                   SUM(last30) AS c_last30, SUM(prev30) AS c_prev30, SUM(prev60) AS c_prev60,
                   SUM(f1) AS c_f1, SUM(f2) AS c_f2
            FROM elig
            GROUP BY 1
        ),
        scored AS (
            SELECT e.client_hash_id, e.content_hash_id, DATE '{t_date}' AS snap_t, c.n_elig,
                   -- label ingredients
                   (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
                   (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
                   (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2,
                   -- past-only features
                   e.imp_90, e.clicks_90,
                   1.0 * e.clicks_90 / NULLIF(e.imp_90, 0) AS ctr_90,
                   1.0 * e.clicks_last30 / NULLIF(e.last30, 0) AS ctr_last30,
                   1.0 * e.pos_sum_90 / NULLIF(e.imp_90, 0) AS pos_90,
                   1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0) AS pos_last30,
                   1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0) AS pos_prev30,
                   (1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0))
                     - (1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0)) AS pos_change,
                   e.last30 AS imp_last30, e.prev30 AS imp_prev30, e.prev60 AS imp_prev60,
                   e.active_days_90,
                   SQRT(GREATEST(1.0 * e.imp_sq_90 / {FEATURE_DAYS} - POWER(1.0 * e.imp_90 / {FEATURE_DAYS}, 2), 0))
                     / NULLIF(1.0 * e.imp_90 / {FEATURE_DAYS}, 0) AS cv_90,
                   LN((e.last30 + 1.0) / (e.prev30 + 1.0)) AS log_mom,
                   LN((e.last30 + 1.0) / (e.prev30 + 1.0))
                     - CASE WHEN c.c_prev30 > 0 THEN LN(1.0 * c.c_last30 / c.c_prev30) END AS log_mom_rel,
                   LN((e.prev30 + 1.0) / (e.prev60 + 1.0)) AS log_mom_prev,
                   LN((e.prev30 + 1.0) / (e.prev60 + 1.0))
                     - CASE WHEN c.c_prev30 > 0 AND c.c_prev60 > 0 THEN LN(1.0 * c.c_prev30 / c.c_prev60) END AS log_mom_prev_rel,
                   DATE '{t_date}' - e.content_created_date AS page_age_days,
                   {static_out}
            FROM elig e
            JOIN cl c ON c.client_hash_id = e.client_hash_id
            WHERE c.n_elig >= {MIN_CLIENT_PAGES}
              AND c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
        )
        SELECT *, CAST(rel <= {REL_MAX} AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER) AS y
        FROM scored
    """)

CACHE_DIR = 'snap_cache'   # local to this Colab runtime; never commit it (no raw exports in the repo)
os.makedirs(CACHE_DIR, exist_ok=True)
expected = label_summary.set_index('snapshot_t')
for t_str in SNAPSHOTS['train']:
    path = f'{CACHE_DIR}/snap_{t_str}.parquet'
    if os.path.exists(path):
        continue                      # already built in this runtime
    build_snapshot(t_str, table='snap_tmp')
    n, pos = con.sql("SELECT COUNT(*), COALESCE(SUM(y), 0) FROM snap_tmp").fetchone()
    # the combined build must reproduce the locked label exactly
    assert (n, pos) == (expected.loc[t_str, 'n_pages'], expected.loc[t_str, 'n_pos']), f'label mismatch at {t_str}'
    con.sql(f"COPY snap_tmp TO '{path}' (FORMAT PARQUET)")
con.sql(f"CREATE OR REPLACE TEMP TABLE train_set AS SELECT * FROM read_parquet('{CACHE_DIR}/snap_*.parquet')")

con.sql("SELECT COUNT(*) AS n_rows, SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate FROM train_set").show()

# Missing values per feature (decides imputation later)
all_feats = PAST_COLS + STATIC_COLS
nulls = con.sql("SELECT " + ", ".join(f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in all_feats)
                + " FROM train_set").df().T
nulls.columns = ['n_missing']
nulls['share_missing'] = (nulls['n_missing'] / con.sql("SELECT COUNT(*) FROM train_set").fetchone()[0]).round(4)
nulls

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────┬────────┬───────────┐
│ n_rows │ n_pos  │ base_rate │
│ int64  │ int128 │  double   │
├────────┼────────┼───────────┤
│ 272078 │  33090 │     0.122 │
└────────┴────────┴───────────┘



,n_missing,share_missing
imp_90,0.0,0.0000
clicks_90,0.0,0.0000
ctr_90,0.0,0.0000
ctr_last30,0.0,0.0000
pos_90,0.0,0.0000
pos_last30,0.0,0.0000
pos_prev30,4581.0,0.0168
pos_change,4581.0,0.0168
imp_last30,0.0,0.0000
imp_prev30,0.0,0.0000


In [ ]:
MIN_CLIENT_PAGES_MODEL = 200   # replaces 100; filtering on n_elig is equivalent to rebuilding with MIN_CLIENT_PAGES = 200

# Unfiltered table, rebuilt from the cached snapshots (safe to re-run)
con.sql(f"CREATE OR REPLACE TEMP TABLE train_set_all AS SELECT * FROM read_parquet('{CACHE_DIR}/snap_*.parquet')")

# What does the 200 rule remove? (a client can move between buckets as it grows)
con.sql("""
    SELECT CASE WHEN n_elig < 200 THEN '100-199' WHEN n_elig < 500 THEN '200-499'
                WHEN n_elig < 2000 THEN '500-1999' ELSE '2000+' END AS client_size_at_t,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(*) AS n_rows, ROUND(AVG(y), 3) AS base_rate
    FROM train_set_all
    GROUP BY 1
    ORDER BY MIN(n_elig)
""").show()

con.sql(f"CREATE OR REPLACE TEMP TABLE train_set AS SELECT * FROM train_set_all WHERE n_elig >= {MIN_CLIENT_PAGES_MODEL}")
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients, SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate FROM train_set").show()

# Does the label rate depend on past relative momentum? decile 1 = fell most, 10 = rose most (deciles within each snapshot)
for scope, cond in [('all 12 snapshots', "TRUE"), ('last 4 snapshots', "snap_t >= DATE '2025-12-31'")]:
    print(scope)
    con.sql(f"""
        SELECT decile, COUNT(*) AS n, ROUND(AVG(y), 3) AS label_rate, ROUND(AVG(log_mom_rel), 2) AS avg_log_mom_rel
        FROM (SELECT y, log_mom_rel,
                     NTILE(10) OVER (PARTITION BY snap_t ORDER BY log_mom_rel) AS decile
              FROM train_set WHERE {cond})
        GROUP BY decile
        ORDER BY decile
    """).show()

# Baselines again on the filtered set
d = con.sql("SELECT snap_t, y, log_mom_rel, log_mom, imp_prev30, imp_last30 FROM train_set").df()
d['rule_rel_momentum'] = -d['log_mom_rel']
d['rule_raw_momentum'] = -d['log_mom']
d['rule_impressions_lost'] = d['imp_prev30'] - d['imp_last30']

def summarize(df, label):
    out = []
    for r in RULES:
        e = eval_rule(df, r)
        out.append({'rule': r, **e[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()})
    print(label)
    print(pd.DataFrame(out).to_string(index=False))

summarize(d, 'Baselines, all 12 training snapshots')
summarize(d[d['snap_t'].astype(str) >= '2025-12-31'], 'Baselines, last 4 snapshots (Dec 2025 to Mar 2026)')

┌──────────────────┬───────────┬────────┬───────────┐
│ client_size_at_t │ n_clients │ n_rows │ base_rate │
│     varchar      │   int64   │ int64  │  double   │
├──────────────────┼───────────┼────────┼───────────┤
│ 100-199          │         8 │   2551 │     0.275 │
│ 200-499          │         5 │   3267 │      0.27 │
│ 500-1999         │         8 │  22545 │     0.199 │
│ 2000+            │         7 │ 243715 │     0.111 │
└──────────────────┴───────────┴────────┴───────────┘

┌────────┬───────────┬────────┬───────────┐
│ n_rows │ n_clients │ n_pos  │ base_rate │
│ int64  │   int64   │ int128 │  double   │
├────────┼───────────┼────────┼───────────┤
│ 269527 │        13 │  32389 │      0.12 │
└────────┴───────────┴────────┴───────────┘

all 12 snapshots
┌────────┬───────┬────────────┬─────────────────┐
│ decile │   n   │ label_rate │ avg_log_mom_rel │
│ int64  │ int64 │   double   │     double      │
├────────┼───────┼────────────┼─────────────────┤
│      1 │ 26958 │      0.166 │

NameError: name 'RULES' is not defined

In [ ]:
# Per-snapshot table after the 200-page client minimum
con.sql("""
    SELECT snap_t, COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients,
           SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate
    FROM train_set
    GROUP BY 1
    ORDER BY 1
""").show(max_rows=20)

# How much of the label is "a recent spike falling back"? (descriptive state at t, past data only; thresholds are exploratory)
for scope, cond in [('all 12 snapshots', "TRUE"), ('last 4 snapshots', "snap_t >= DATE '2025-12-31'")]:
    print(scope)
    con.sql(f"""
        WITH s AS (
            SELECT y,
                   CASE WHEN imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60) THEN 'a: spike (last 30d >= 2x both earlier months)'
                        WHEN imp_last30 <= 0.8 * imp_prev30 AND imp_prev30 <= 0.8 * imp_prev60 THEN 'b: steady decline (two falls of 20%+)'
                        ELSE 'c: other' END AS state_at_t
            FROM train_set
            WHERE {cond}
        )
        SELECT state_at_t, COUNT(*) AS n,
               ROUND(1.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS share_of_rows,
               ROUND(AVG(y), 3) AS label_rate,
               ROUND(1.0 * SUM(y) / SUM(SUM(y)) OVER (), 3) AS share_of_positives
        FROM s
        GROUP BY 1
        ORDER BY 1
    """).show()

CatalogException: Catalog Error: Table with name train_set does not exist!
Did you mean "pg_attribute"?

In [ ]:
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"   # new or spiking at t (past data only)

con.sql(f"""
    CREATE OR REPLACE TEMP TABLE train_set_flagged AS
    SELECT *, ({SPIKE_RULE}) AS is_spike
    FROM train_set_all
    WHERE n_elig >= {MIN_CLIENT_PAGES_MODEL}
""")
con.sql("CREATE OR REPLACE TEMP TABLE train_spike AS SELECT * FROM train_set_flagged WHERE is_spike")
con.sql("CREATE OR REPLACE TEMP TABLE train_set AS SELECT * FROM train_set_flagged WHERE NOT is_spike")

# What leaves the decline model, and what stays
con.sql("""
    SELECT is_spike, COUNT(*) AS n_rows, SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate
    FROM train_set_flagged
    GROUP BY 1
    ORDER BY 1
""").show()

con.sql("""
    SELECT snap_t, COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients,
           SUM(y) AS n_pos, ROUND(AVG(y), 3) AS base_rate
    FROM train_set
    GROUP BY 1
    ORDER BY 1
""").show(max_rows=20)

# Label rate by past relative momentum on the remaining pages (decile 1 = fell most)
con.sql("""
    SELECT decile, COUNT(*) AS n, ROUND(AVG(y), 3) AS label_rate, ROUND(AVG(log_mom_rel), 2) AS avg_log_mom_rel
    FROM (SELECT y, log_mom_rel,
                 NTILE(10) OVER (PARTITION BY snap_t ORDER BY log_mom_rel) AS decile
          FROM train_set)
    GROUP BY decile
    ORDER BY decile
""").show()

# Baselines again on the non-spike pages
d = con.sql("SELECT snap_t, y, log_mom_rel, log_mom, imp_prev30, imp_last30 FROM train_set").df()
d['rule_rel_momentum'] = -d['log_mom_rel']
d['rule_raw_momentum'] = -d['log_mom']
d['rule_impressions_lost'] = d['imp_prev30'] - d['imp_last30']
summarize(d, 'Baselines on non-spike pages, all 12 training snapshots')
summarize(d[d['snap_t'].astype(str) >= '2025-12-31'], 'Baselines on non-spike pages, last 4 snapshots')

CatalogException: Catalog Error: Table with name train_set_all does not exist!
Did you mean "sqlite_temp_master"?

LINE 4:     FROM train_set_all
                 ^

In [14]:
import os
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import average_precision_score

if 'con' not in globals():
    con = duckdb.connect()

# Same constants as the Section 2 setup cell
LABEL_DAYS = 30
FEATURE_DAYS = 90
SNAPSHOTS = {
    'train': ['2025-04-30', '2025-05-31', '2025-06-30', '2025-07-31', '2025-08-31', '2025-09-30',
              '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31'],
    'validation': '2026-04-30',
    'test': '2026-05-31',
    'final_list': '2026-06-30',
}
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
K_LIST = [20, 50]
CACHE_DIR = 'snap_cache'                 # use the Drive path here if you saved the cache to Drive
MIN_CLIENT_PAGES_MODEL = 200
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"

n_files = len([f for f in os.listdir(CACHE_DIR) if f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{CACHE_DIR} has {n_files} snapshot files, expected 12: re-run the setup, label and features cells'

# Rebuild the modeling table from the cached snapshots: 200-page client minimum, then drop new or spiking pages
con.sql(f"CREATE OR REPLACE TEMP TABLE train_set_all AS SELECT * FROM read_parquet('{CACHE_DIR}/snap_*.parquet')")
con.sql(f"""
    CREATE OR REPLACE TEMP TABLE train_set_flagged AS
    SELECT *, ({SPIKE_RULE}) AS is_spike
    FROM train_set_all
    WHERE n_elig >= {MIN_CLIENT_PAGES_MODEL}
""")
con.sql("CREATE OR REPLACE TEMP TABLE train_spike AS SELECT * FROM train_set_flagged WHERE is_spike")
con.sql("CREATE OR REPLACE TEMP TABLE train_set AS SELECT * FROM train_set_flagged WHERE NOT is_spike")

n_rows, n_pos = con.sql("SELECT COUNT(*), SUM(y) FROM train_set").fetchone()
print(f"train_set: {n_rows:,} rows, {n_pos:,} positives, base rate {n_pos / n_rows:.3f}")
assert (n_rows, n_pos) == (219258, 22429), 'does not match the earlier run (219,258 rows, 22,429 positives)'

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

train_set: 219,258 rows, 22,429 positives, base rate 0.102


In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

LOG_COLS = ['imp_90', 'clicks_90', 'imp_last30', 'imp_prev30', 'imp_prev60']   # count features, logged for the linear model

data = con.sql(f"SELECT snap_t, y, {', '.join(PAST_COLS)} FROM train_set").df()
data['snap_t'] = pd.to_datetime(data['snap_t'])

def make_X(df, log=False):
    X = df[PAST_COLS].copy()
    for c in X.columns:
        if pd.api.types.is_timedelta64_dtype(X[c]):
            X[c] = X[c].dt.days          # in case a date difference arrives as an interval
    X = X.astype(float)
    if log:
        for c in LOG_COLS:
            X[c] = np.log1p(X[c])
    return X

def make_hgb(seed=0):
    # hyperparameters fixed in advance, not tuned on any evaluation snapshot
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=200, l2_regularization=1.0,
                                          early_stopping=False, random_state=seed)

def fit_score(train, test, which, seed=0):
    y_tr = train['y'].values
    if which == 'hgb':
        m = make_hgb(seed).fit(make_X(train), y_tr)
        return m.predict_proba(make_X(test))[:, 1]
    if which == 'logit':
        m = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(),
                          LogisticRegression(C=0.1, max_iter=1000))
        m.fit(make_X(train, log=True), y_tr)
        return m.predict_proba(make_X(test, log=True))[:, 1]
    if which == 'hgb_shuffled':   # control: same model, labels shuffled inside the training fold
        y_sh = np.random.default_rng(seed).permutation(y_tr)
        m = make_hgb(seed).fit(make_X(train), y_sh)
        return m.predict_proba(make_X(test))[:, 1]

# Walk-forward on the last 6 training snapshots. Embargo: a training snapshot's label window
# (30 days) must end on or before the evaluation snapshot date.
EVAL_SNAPS = SNAPSHOTS['train'][-6:]
parts = []
for t_eval in EVAL_SNAPS:
    te_t = pd.Timestamp(t_eval)
    train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]
    test = data[data['snap_t'] == te_t].copy()
    if len(train) == 0 or len(test) == 0 or train['y'].nunique() < 2:
        print(f"{t_eval}: skipped (no usable training rows)")
        continue
    print(f"{t_eval}: train rows {len(train):,} (up to {train['snap_t'].max().date()}), test rows {len(test):,}")
    for which in ['hgb', 'logit', 'hgb_shuffled']:
        test[which] = fit_score(train, test, which)
    test['rule_rel_momentum'] = -test['log_mom_rel']
    test['rule_raw_momentum'] = -test['log_mom']
    test['rule_impressions_lost'] = test['imp_prev30'] - test['imp_last30']
    parts.append(test)
res = pd.concat(parts)

SCORES = ['hgb', 'logit', 'hgb_shuffled', 'rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']
summary = []
for s in SCORES:
    e = eval_rule(res, s)
    summary.append({'score': s, **e[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()})
print()
print(pd.DataFrame(summary).to_string(index=False))

# Per-snapshot detail for the main model
eval_rule(res, 'hgb')

2025-10-31: train rows 35,090 (up to 2025-09-30), test rows 13,547
2025-11-30: train rows 48,637 (up to 2025-10-31), test rows 17,738
2025-12-31: train rows 66,375 (up to 2025-11-30), test rows 27,117
2026-01-31: train rows 93,492 (up to 2025-12-31), test rows 30,498
2026-02-28: train rows 93,492 (up to 2025-12-31), test rows 41,468
2026-03-31: train rows 165,458 (up to 2026-02-28), test rows 53,800

                score  p@20  lift@20  p@50  lift@50  ap_lift
                  hgb 0.508    7.633 0.423    6.255    2.282
                logit 0.450    7.003 0.447    6.758    1.962
         hgb_shuffled 0.183    2.895 0.147    2.162    1.133
    rule_rel_momentum 0.250    4.028 0.237    3.882    1.337
    rule_raw_momentum 0.233    3.285 0.220    3.430    1.342
rule_impressions_lost 0.258    4.217 0.243    3.897    1.288


,snap_t,n,base_rate,p@20,lift@20,p@50,lift@50,ap,ap_lift
0,2025-10-31,13547,0.048,0.40,8.38,0.26,5.44,0.118,2.47
1,2025-11-30,17738,0.040,0.05,1.27,0.12,3.04,0.092,2.32
2,2025-12-31,27117,0.073,0.85,11.64,0.64,8.77,0.189,2.59
3,2026-01-31,30498,0.047,0.70,14.93,0.54,11.52,0.112,2.40
4,2026-02-28,41468,0.080,0.50,6.29,0.44,5.53,0.178,2.23
5,2026-03-31,53800,0.167,0.55,3.29,0.54,3.23,0.280,1.68


In [16]:
from sklearn.inspection import permutation_importance

def folds(t_list):
    for t_eval in t_list:
        te_t = pd.Timestamp(t_eval)
        train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t].reset_index(drop=True)
        test = data[data['snap_t'] == te_t].copy()
        if len(train) == 0 or len(test) == 0 or train['y'].nunique() < 2:
            continue
        yield t_eval, train, test

# 1. A proper null model: labels shuffled WITHIN each training snapshot, 3 seeds, same folds as before
def shuffle_within_snapshot(train, seed):
    y_new = train['y'].values.copy()
    rng = np.random.default_rng(seed)
    for _, idx in train.groupby('snap_t').indices.items():
        y_new[idx] = rng.permutation(y_new[idx])
    return y_new

null_parts = {seed: [] for seed in range(3)}
for t_eval, train, test in folds(EVAL_SNAPS):
    for seed in range(3):
        m = make_hgb(seed).fit(make_X(train), shuffle_within_snapshot(train, seed))
        ts = test.copy()
        ts['null'] = m.predict_proba(make_X(ts))[:, 1]
        null_parts[seed].append(ts[['snap_t', 'y', 'null']])
null_rows = [eval_rule(pd.concat(p), 'null')[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean()
             for p in null_parts.values()]
print('Null model (labels shuffled within each snapshot), mean over folds, one row per seed:')
print(pd.DataFrame(null_rows).round(3).to_string(index=False))

# 2. What drives the model, and what do its top-20 pages look like? (two folds)
PROFILE = ['imp_last30', 'imp_prev30', 'imp_90', 'active_days_90', 'cv_90', 'log_mom_rel',
           'pos_last30', 'ctr_last30', 'page_age_days']
for t_eval, train, test in folds(['2025-12-31', '2026-03-31']):
    m = make_hgb(0).fit(make_X(train), train['y'].values)
    X_te = make_X(test)
    test['score'] = m.predict_proba(X_te)[:, 1]
    imp = permutation_importance(m, X_te, test['y'].values, scoring='average_precision',
                                 n_repeats=3, random_state=0, n_jobs=1)
    print(f"\nFold {t_eval}: top permutation importances (drop in average precision)")
    print(pd.Series(imp.importances_mean, index=PAST_COLS).sort_values(ascending=False).head(8).round(4).to_string())
    top_idx = test['score'].nlargest(20).index
    prof = pd.DataFrame({'top20_median': X_te.loc[top_idx, PROFILE].median(),
                         'all_pages_median': X_te[PROFILE].median()}).round(2)
    print(f"Fold {t_eval}: top-20 pages vs all pages (medians)")
    print(prof.to_string())

Null model (labels shuffled within each snapshot), mean over folds, one row per seed:
 p@20  lift@20  p@50  lift@50  ap_lift
0.133    1.722 0.127    1.493    1.077
0.075    1.083 0.093    1.387    1.103
0.233    3.892 0.180    2.745    1.152

Fold 2025-12-31: top permutation importances (drop in average precision)
cv_90               0.0747
pos_last30          0.0227
active_days_90      0.0162
log_mom_prev_rel    0.0124
imp_last30          0.0100
log_mom_rel         0.0047
imp_90              0.0043
ctr_90              0.0039
Fold 2025-12-31: top-20 pages vs all pages (medians)
                top20_median  all_pages_median
imp_last30            180.50            686.00
imp_prev30            236.50            738.00
imp_90                460.00           2068.00
active_days_90         45.00             90.00
cv_90                   1.48              0.55
log_mom_rel            -0.65             -0.04
pos_last30             68.70              6.62
ctr_last30              0.00           

In [17]:
FOLDS4 = SNAPSHOTS['train'][-4:]   # Dec 2025 to Mar 2026, the most recent regime
CAT_COLS = ['content_type', 'main_intent', 'competition_level']

def load(table, where='TRUE'):
    cols = ['client_hash_id', 'snap_t', 'y'] + PAST_COLS + STATIC_COLS
    d = con.sql(f"SELECT {', '.join(cols)} FROM {table} WHERE {where}").df()
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    return d

data = load('train_set')                                                  # main universe (also used by the earlier cells)
data_spike = load('train_set_flagged')                                    # includes new or spiking pages
data_min100 = load('train_set_all', f"n_elig >= 100 AND NOT ({SPIKE_RULE})")  # client minimum 100 instead of 200

CAT_MAPS = {c: {v: i for i, v in enumerate(sorted(data[c].dropna().unique()))} for c in CAT_COLS}

def make_X2(df, cols):
    X = pd.DataFrame(index=df.index)
    for c in cols:
        if c in CAT_COLS:
            X[c] = df[c].map(CAT_MAPS[c]).astype(float)
        else:
            s = df[c]
            X[c] = s.dt.days if pd.api.types.is_timedelta64_dtype(s) else s
    return X.astype(float)

def fit_predict(train, test, cols, seed=0):
    cat_mask = [c in CAT_COLS for c in cols]
    kw = dict(categorical_features=cat_mask) if any(cat_mask) else {}
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                       min_samples_leaf=200, l2_regularization=1.0,
                                       early_stopping=False, random_state=seed, **kw)
    m.fit(make_X2(train, cols), train['y'].values)
    return m.predict_proba(make_X2(test, cols))[:, 1]

def walk_forward(df, cols, fold_list, train_filter=None):
    out = []
    for t_eval in fold_list:
        te_t = pd.Timestamp(t_eval)
        train = df[df['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]      # embargo
        if train_filter is not None:
            train = train[train_filter(train)]
        test = df[df['snap_t'] == te_t].copy()
        if len(train) == 0 or len(test) == 0 or train['y'].nunique() < 2:
            continue
        test['score'] = fit_predict(train, test, cols)
        out.append(test)
    return pd.concat(out)

def add_rules(df):
    df['rule_rel_momentum'] = -df['log_mom_rel']
    df['rule_raw_momentum'] = -df['log_mom']
    df['rule_impressions_lost'] = df['imp_prev30'] - df['imp_last30']
    return df

def row(name, scored, col='score'):
    e = eval_rule(scored, col)
    return {'variant': name, 'rows': len(scored),
            **e[['base_rate', 'p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()}

rows = []
main = add_rules(walk_forward(data, PAST_COLS, FOLDS4))
rows.append(row('main (past-only, last 4 folds)', main))
rows.append(row('+ static content fields', walk_forward(data, PAST_COLS + STATIC_COLS, FOLDS4)))
rows.append(row('client minimum 100 instead of 200', walk_forward(data_min100, PAST_COLS, FOLDS4)))
rows.append(row('with new or spiking pages', walk_forward(data_spike, PAST_COLS, FOLDS4)))
rows.append(row('trained without the first six snapshots',
                walk_forward(data, PAST_COLS, FOLDS4, train_filter=lambda t: t['snap_t'] >= pd.Timestamp('2025-10-31'))))

# Client-held-out: two groups of clients (alternating by size rank); each group is scored by a model trained on the other
sizes = data.groupby('client_hash_id').size().sort_values(ascending=False)
grp = {c: ('A' if i % 2 == 0 else 'B') for i, c in enumerate(sizes.index)}
data['grp'] = data['client_hash_id'].map(grp)
held = []
for t_eval in FOLDS4:
    te_t = pd.Timestamp(t_eval)
    tr_all = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]
    te_all = data[data['snap_t'] == te_t]
    for g_train, g_test in [('A', 'B'), ('B', 'A')]:
        tr, te = tr_all[tr_all['grp'] == g_train], te_all[te_all['grp'] == g_test].copy()
        if len(tr) and len(te) and tr['y'].nunique() == 2:
            te['score'] = fit_predict(tr, te, PAST_COLS)
            held.append(te)
rows.append(row('client-held-out (trained on the other clients)', pd.concat(held)))

print(pd.DataFrame(rows).to_string(index=False))

# Does it work where more impressions are at stake? Same predictions, pages with >= 1,000 impressions in the last 30 days
print()
for floor in [1000]:
    sl = main[main['imp_last30'] >= floor]
    srows = [row(f'model, last30 >= {floor}', sl, 'score')]
    for r in ['rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']:
        srows.append(row(f'{r}, last30 >= {floor}', sl, r))
    print(pd.DataFrame(srows).to_string(index=False))

                                       variant   rows  base_rate  p@20  lift@20  p@50  lift@50  ap_lift
                main (past-only, last 4 folds) 152883      0.092 0.650    9.038 0.540    7.262    2.225
                       + static content fields 152883      0.092 0.725    9.815 0.550    7.162    2.330
             client minimum 100 instead of 200 154034      0.093 0.625    8.565 0.530    7.055    2.170
                     with new or spiking pages 192833      0.114 0.600    6.248 0.610    6.055    2.323
       trained without the first six snapshots 152883      0.092 0.412    5.800 0.425    5.700    2.310
client-held-out (trained on the other clients) 152883      0.092 0.475    6.685 0.480    6.375    2.128

                              variant  rows  base_rate  p@20  lift@20  p@50  lift@50  ap_lift
                model, last30 >= 1000 69940      0.072 0.300    4.130 0.285    3.800    2.190
    rule_rel_momentum, last30 >= 1000 69940      0.072 0.250    4.310 0.195    3.07

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [18]:
%pip -q install duckdb huggingface_hub

import os, gc, getpass
import numpy as np
import pandas as pd
import duckdb

HF_TOKEN = globals().get('HF_TOKEN') or os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception as e:
        print('No stored token found:', type(e).__name__)
HF_TOKEN = (HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')).strip()

def connect_hf():
    """Fresh warehouse connection with a memory cap and disk spill, so big queries slow down instead of crashing."""
    c = duckdb.connect()
    os.makedirs('duck_tmp', exist_ok=True)
    c.execute("SET memory_limit='5GB'")
    c.execute("SET temp_directory='duck_tmp'")
    c.execute("SET threads=2")
    c.execute("SET preserve_insertion_order=false")
    c.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    return c

con = connect_hf()

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# ---- locked design constants ----
LABEL_DAYS = 30
FEATURE_DAYS = 90
FLOOR = 100
MIN_CLIENT_PAGES = 100            # used while building snapshots; the model universe then keeps clients with >= 200
MIN_CLIENT_PAGES_MODEL = 200
REL_MAX = 0.5
K_LIST = [20, 50]
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"
SNAPSHOTS = {
    'train': ['2025-04-30', '2025-05-31', '2025-06-30', '2025-07-31', '2025-08-31', '2025-09-30',
              '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31'],
    'validation': '2026-04-30',
    'test': '2026-05-31',
    'final_list': '2026-06-30',
}
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
CACHE_DIR = 'snap_cache'
EVAL_DIR = 'eval_cache'          # validation and test snapshots are cached here, away from the training files
os.makedirs(EVAL_DIR, exist_ok=True)

n_train_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
print('training snapshot files found:', n_train_files, '(expected 12)')
print('evaluation snapshot files found:', sorted(os.listdir(EVAL_DIR)))

training snapshot files found: 12 (expected 12)
evaluation snapshot files found: []


In [19]:
# Snapshot builder (identical to the features cell)
def build_snapshot(t_date, table='snap'):
    """Labels (same rules as build_labels) plus past-only features for one snapshot t.
    Features use report_date <= t only; the label uses report_date > t only."""
    t = pd.Timestamp(t_date)
    L = LABEL_DAYS
    half = L // 2
    days = pd.date_range(t - pd.Timedelta(days=FEATURE_DAYS - 1), t + pd.Timedelta(days=L))
    months_sql = ', '.join(f"'{m}'" for m in sorted({d.strftime('%Y-%m') for d in days}))
    static_sql = ', '.join(f"d.{c}" for c in STATIC_COLS)
    static_out = ', '.join(f"e.{c}" for c in STATIC_COLS)
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id,
                   gsc_impressions, gsc_clicks, gsc_sum_position
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {FEATURE_DAYS}
              AND report_date <= DATE '{t_date}' + {L}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   -- label side: strictly after t
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2,
                   -- feature side: up to and including t
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS imp_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_clicks ELSE 0 END) AS clicks_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_sum_position ELSE 0 END) AS pos_sum_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions * gsc_impressions ELSE 0 END) AS imp_sq_90,
                   COUNT(*) FILTER (WHERE report_date <= DATE '{t_date}' AND gsc_impressions > 0) AS active_days_90,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_clicks ELSE 0 END) AS clicks_last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' AND report_date > DATE '{t_date}' - {L}
                            THEN gsc_sum_position ELSE 0 END) AS pos_sum_last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_impressions ELSE 0 END) AS prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_sum_position ELSE 0 END) AS pos_sum_prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {2*L} THEN gsc_impressions ELSE 0 END) AS prev60
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*, d.content_created_date, {static_sql}
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, COUNT(*) AS n_elig,
                   SUM(last30) AS c_last30, SUM(prev30) AS c_prev30, SUM(prev60) AS c_prev60,
                   SUM(f1) AS c_f1, SUM(f2) AS c_f2
            FROM elig
            GROUP BY 1
        ),
        scored AS (
            SELECT e.client_hash_id, e.content_hash_id, DATE '{t_date}' AS snap_t, c.n_elig,
                   -- label ingredients
                   (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
                   (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
                   (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2,
                   -- past-only features
                   e.imp_90, e.clicks_90,
                   1.0 * e.clicks_90 / NULLIF(e.imp_90, 0) AS ctr_90,
                   1.0 * e.clicks_last30 / NULLIF(e.last30, 0) AS ctr_last30,
                   1.0 * e.pos_sum_90 / NULLIF(e.imp_90, 0) AS pos_90,
                   1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0) AS pos_last30,
                   1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0) AS pos_prev30,
                   (1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0))
                     - (1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0)) AS pos_change,
                   e.last30 AS imp_last30, e.prev30 AS imp_prev30, e.prev60 AS imp_prev60,
                   e.active_days_90,
                   SQRT(GREATEST(1.0 * e.imp_sq_90 / {FEATURE_DAYS} - POWER(1.0 * e.imp_90 / {FEATURE_DAYS}, 2), 0))
                     / NULLIF(1.0 * e.imp_90 / {FEATURE_DAYS}, 0) AS cv_90,
                   LN((e.last30 + 1.0) / (e.prev30 + 1.0)) AS log_mom,
                   LN((e.last30 + 1.0) / (e.prev30 + 1.0))
                     - CASE WHEN c.c_prev30 > 0 THEN LN(1.0 * c.c_last30 / c.c_prev30) END AS log_mom_rel,
                   LN((e.prev30 + 1.0) / (e.prev60 + 1.0)) AS log_mom_prev,
                   LN((e.prev30 + 1.0) / (e.prev60 + 1.0))
                     - CASE WHEN c.c_prev30 > 0 AND c.c_prev60 > 0 THEN LN(1.0 * c.c_prev30 / c.c_prev60) END AS log_mom_prev_rel,
                   DATE '{t_date}' - e.content_created_date AS page_age_days,
                   {static_out}
            FROM elig e
            JOIN cl c ON c.client_hash_id = e.client_hash_id
            WHERE c.n_elig >= {MIN_CLIENT_PAGES}
              AND c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
        )
        SELECT *, CAST(rel <= {REL_MAX} AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER) AS y
        FROM scored
    """)

In [20]:
def build_and_cache(t_str, name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    if os.path.exists(path):
        print(name, 'already cached, skipping the build')
        return
    build_snapshot(t_str, table=f'snap_{name}')
    con.sql(f"COPY snap_{name} TO '{path}' (FORMAT PARQUET)")
    con.sql(f"DROP TABLE snap_{name}")
    gc.collect()
    print(name, 'built and saved to', path)

build_and_cache(SNAPSHOTS['validation'], 'validation')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

validation built and saved to eval_cache/eval_validation.parquet


In [21]:
build_and_cache(SNAPSHOTS['test'], 'test')

# Warehouse work is finished for the modeling cells: close this connection to free its memory
con.close()
gc.collect()
print('warehouse connection closed; the modeling cells read only the saved files')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test built and saved to eval_cache/eval_test.parquet
warehouse connection closed; the modeling cells read only the saved files


In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score

mcon = duckdb.connect()          # local connection for the saved files only (no warehouse access)

# ---- helpers ----
LOG_COLS = ['imp_90', 'clicks_90', 'imp_last30', 'imp_prev30', 'imp_prev60']

def make_X(df, log=False):
    X = df[PAST_COLS].copy()
    for c in X.columns:
        if pd.api.types.is_timedelta64_dtype(X[c]):
            X[c] = X[c].dt.days
    X = X.astype(float)
    if log:
        for c in LOG_COLS:
            X[c] = np.log1p(X[c])
    return X

def make_hgb(seed=0):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=200, l2_regularization=1.0,
                                          early_stopping=False, random_state=seed)

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def shuffle_within_snapshot(train, seed):
    y_new = train['y'].values.copy()
    rng = np.random.default_rng(seed)
    for _, idx in train.reset_index(drop=True).groupby('snap_t').indices.items():
        y_new[idx] = rng.permutation(y_new[idx])
    return y_new

def score_all(train, evald):
    out = evald.copy()
    out['hgb'] = make_hgb(0).fit(make_X(train), train['y'].values).predict_proba(make_X(out))[:, 1]
    lg = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(),
                       LogisticRegression(C=0.1, max_iter=1000))
    out['logit'] = lg.fit(make_X(train, log=True), train['y'].values).predict_proba(make_X(out, log=True))[:, 1]
    train = train.reset_index(drop=True)
    for s in range(3):   # null models: labels shuffled within each training snapshot
        out[f'null{s}'] = make_hgb(s).fit(make_X(train), shuffle_within_snapshot(train, s)).predict_proba(make_X(out))[:, 1]
    out['rule_rel_momentum'] = -out['log_mom_rel']
    out['rule_raw_momentum'] = -out['log_mom']
    out['rule_impressions_lost'] = out['imp_prev30'] - out['imp_last30']
    return out

SCORES = ['hgb', 'logit', 'null0', 'null1', 'null2', 'rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']

def report(title, df):
    print('=' * 8, title, '=' * 8)
    tab = pd.concat([eval_rule(df, c).assign(score=c) for c in SCORES])
    print(tab[['score', 'n', 'base_rate', 'p@20', 'lift@20', 'p@50', 'lift@50', 'ap', 'ap_lift']].to_string(index=False))
    sl = df[df['imp_last30'] >= 1000]
    print(f"\nPages with >= 1,000 impressions in the last 30 days (n={len(sl):,}):")
    tab = pd.concat([eval_rule(sl, c).assign(score=c) for c in ['hgb', 'logit', 'rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']])
    print(tab[['score', 'n', 'base_rate', 'p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].to_string(index=False))
    rows = []
    sizes = df.groupby('client_hash_id').size().sort_values(ascending=False)
    for rank, (c, n) in enumerate(sizes.items(), 1):
        g = df[df['client_hash_id'] == c]
        base = g['y'].mean()
        if g['y'].sum() == 0 or base == 0:
            rows.append({'client_rank': rank, 'n': n, 'base_rate': round(base, 3)})
            continue
        best_rule = max(average_precision_score(g['y'], g[r]) for r in ['rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost'])
        rows.append({'client_rank': rank, 'n': n, 'base_rate': round(base, 3),
                     'ap_lift_model': round(average_precision_score(g['y'], g['hgb']) / base, 2),
                     'ap_lift_best_rule': round(best_rule / base, 2)})
    print('\nPer client (ranked by size, ids not shown):')
    print(pd.DataFrame(rows).to_string(index=False))
    print()

In [23]:
cols_sql = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
UNIVERSE = f"n_elig >= {MIN_CLIENT_PAGES_MODEL} AND NOT ({SPIKE_RULE})"

n_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{CACHE_DIR} has {n_files} snapshot files, expected 12'
train_all = mcon.sql(f"SELECT {cols_sql} FROM read_parquet('{CACHE_DIR}/snap_*.parquet') WHERE {UNIVERSE}").df()
train_all['snap_t'] = pd.to_datetime(train_all['snap_t'])
assert len(train_all) == 219258, 'training table does not match the earlier run'

def load_eval(name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    assert os.path.exists(path), f'{path} is missing: run the build cells first'
    d = mcon.sql(f"SELECT {cols_sql} FROM read_parquet('{path}') WHERE {UNIVERSE}").df()
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    return d

val = load_eval('validation')
test = load_eval('test')
print(f"training: {len(train_all):,} rows, base rate {train_all['y'].mean():.3f}")
print(f"validation {SNAPSHOTS['validation']}: {len(val):,} rows, base rate {val['y'].mean():.3f}")
print(f"test {SNAPSHOTS['test']}: {len(test):,} rows, base rate {test['y'].mean():.3f}")

training: 219,258 rows, base rate 0.102
validation 2026-04-30: 60,463 rows, base rate 0.169
test 2026-05-31: 75,873 rows, base rate 0.221


In [24]:
# Model, hyperparameters, universe and metrics were all fixed before this cell. Nothing is chosen from these results.
val_scored = score_all(train_all, val)
report(f"VALIDATION {SNAPSHOTS['validation']} (trained on the 12 training snapshots)", val_scored)

======== VALIDATION 2026-04-30 (trained on the 12 training snapshots) ========
                score     n  base_rate  p@20  lift@20  p@50  lift@50    ap  ap_lift
                  hgb 60463      0.169  0.60     3.56  0.50     2.96 0.313     1.86
                logit 60463      0.169  0.65     3.85  0.46     2.73 0.271     1.60
                null0 60463      0.169  0.05     0.30  0.04     0.24 0.140     0.83
                null1 60463      0.169  0.15     0.89  0.26     1.54 0.146     0.86
                null2 60463      0.169  0.10     0.59  0.22     1.30 0.145     0.86
    rule_rel_momentum 60463      0.169  0.20     1.19  0.14     0.83 0.188     1.11
    rule_raw_momentum 60463      0.169  0.25     1.48  0.14     0.83 0.211     1.25
rule_impressions_lost 60463      0.169  0.25     1.48  0.28     1.66 0.193     1.15

Pages with >= 1,000 impressions in the last 30 days (n=26,391):
                score     n  base_rate  p@20  lift@20  p@50  lift@50  ap_lift
                  hgb 

In [25]:
# Run once. Do not change anything in the design after looking at this output.
test_scored = score_all(pd.concat([train_all, val]), test)
report(f"SEALED TEST {SNAPSHOTS['test']} (trained on the 12 training snapshots + validation)", test_scored)

======== SEALED TEST 2026-05-31 (trained on the 12 training snapshots + validation) ========
                score     n  base_rate  p@20  lift@20  p@50  lift@50    ap  ap_lift
                  hgb 75873      0.221  0.60     2.71  0.62     2.80 0.338     1.53
                logit 75873      0.221  0.35     1.58  0.24     1.08 0.314     1.42
                null0 75873      0.221  0.20     0.90  0.16     0.72 0.198     0.89
                null1 75873      0.221  0.20     0.90  0.14     0.63 0.200     0.90
                null2 75873      0.221  0.10     0.45  0.10     0.45 0.205     0.93
    rule_rel_momentum 75873      0.221  0.15     0.68  0.18     0.81 0.226     1.02
    rule_raw_momentum 75873      0.221  0.20     0.90  0.14     0.63 0.241     1.09
rule_impressions_lost 75873      0.221  0.15     0.68  0.22     0.99 0.239     1.08

Pages with >= 1,000 impressions in the last 30 days (n=29,656):
                score     n  base_rate  p@20  lift@20  p@50  lift@50  ap_lift
        

In [26]:
# Needs train_all, val, test and the helpers from the modeling cells. It opens its own warehouse connection.
import json
con = connect_hf()


def build_features_only(t_date, table='feat_only'):
    """Same past-only features as build_snapshot, but the warehouse is cut at t: no row after t is read."""
    t = pd.Timestamp(t_date)
    L = LABEL_DAYS
    days = pd.date_range(t - pd.Timedelta(days=FEATURE_DAYS - 1), t)
    months_sql = ', '.join(f"'{m}'" for m in sorted({d.strftime('%Y-%m') for d in days}))
    static_sql = ', '.join(f"d.{c}" for c in STATIC_COLS)
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id,
                   gsc_impressions, gsc_clicks, gsc_sum_position
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {FEATURE_DAYS}
              AND report_date <= DATE '{t_date}'
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) AS imp_90,
                   SUM(gsc_clicks) AS clicks_90,
                   SUM(gsc_sum_position) AS pos_sum_90,
                   SUM(gsc_impressions * gsc_impressions) AS imp_sq_90,
                   COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_90,
                   SUM(CASE WHEN report_date > DATE '{t_date}' - {L} THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date > DATE '{t_date}' - {L} THEN gsc_clicks ELSE 0 END) AS clicks_last30,
                   SUM(CASE WHEN report_date > DATE '{t_date}' - {L} THEN gsc_sum_position ELSE 0 END) AS pos_sum_last30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_impressions ELSE 0 END) AS prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {L} AND report_date > DATE '{t_date}' - {2*L}
                            THEN gsc_sum_position ELSE 0 END) AS pos_sum_prev30,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' - {2*L} THEN gsc_impressions ELSE 0 END) AS prev60
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*, d.content_created_date, {static_sql}
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, COUNT(*) AS n_elig,
                   SUM(last30) AS c_last30, SUM(prev30) AS c_prev30, SUM(prev60) AS c_prev60
            FROM elig
            GROUP BY 1
        )
        SELECT e.client_hash_id, e.content_hash_id, DATE '{t_date}' AS snap_t, c.n_elig,
               e.imp_90, e.clicks_90,
               1.0 * e.clicks_90 / NULLIF(e.imp_90, 0) AS ctr_90,
               1.0 * e.clicks_last30 / NULLIF(e.last30, 0) AS ctr_last30,
               1.0 * e.pos_sum_90 / NULLIF(e.imp_90, 0) AS pos_90,
               1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0) AS pos_last30,
               1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0) AS pos_prev30,
               (1.0 * e.pos_sum_last30 / NULLIF(e.last30, 0))
                 - (1.0 * e.pos_sum_prev30 / NULLIF(e.prev30, 0)) AS pos_change,
               e.last30 AS imp_last30, e.prev30 AS imp_prev30, e.prev60 AS imp_prev60,
               e.active_days_90,
               SQRT(GREATEST(1.0 * e.imp_sq_90 / {FEATURE_DAYS} - POWER(1.0 * e.imp_90 / {FEATURE_DAYS}, 2), 0))
                 / NULLIF(1.0 * e.imp_90 / {FEATURE_DAYS}, 0) AS cv_90,
               LN((e.last30 + 1.0) / (e.prev30 + 1.0)) AS log_mom,
               LN((e.last30 + 1.0) / (e.prev30 + 1.0))
                 - CASE WHEN c.c_prev30 > 0 THEN LN(1.0 * c.c_last30 / c.c_prev30) END AS log_mom_rel,
               LN((e.prev30 + 1.0) / (e.prev60 + 1.0)) AS log_mom_prev,
               LN((e.prev30 + 1.0) / (e.prev60 + 1.0))
                 - CASE WHEN c.c_prev30 > 0 AND c.c_prev60 > 0 THEN LN(1.0 * c.c_prev30 / c.c_prev60) END AS log_mom_prev_rel,
               DATE '{t_date}' - e.content_created_date AS page_age_days
        FROM elig e
        JOIN cl c ON c.client_hash_id = e.client_hash_id
    """)

# ---- 1. Leakage test on the real warehouse -------------------------------------------------------------
# Rebuild the 2026-03-31 features from data cut at t and compare them with the features stored in the training cache.
CHECK_T = '2026-03-31'
build_features_only(CHECK_T, 'feat_check')
stored = f"read_parquet('{CACHE_DIR}/snap_{CHECK_T}.parquet')"
mism = ', '.join(
    f"SUM(CASE WHEN s.{c} IS DISTINCT FROM f.{c} AND NOT (s.{c} IS NOT NULL AND f.{c} IS NOT NULL AND ABS(s.{c} - f.{c}) < 1e-9) THEN 1 ELSE 0 END) AS {c}"
    for c in PAST_COLS)
n_matched = con.sql(f"SELECT COUNT(*) FROM {stored} s JOIN feat_check f ON s.client_hash_id = f.client_hash_id AND s.content_hash_id = f.content_hash_id").fetchone()[0]
n_stored = con.sql(f"SELECT COUNT(*) FROM {stored}").fetchone()[0]
bad = con.sql(f"SELECT {mism} FROM {stored} s JOIN feat_check f ON s.client_hash_id = f.client_hash_id AND s.content_hash_id = f.content_hash_id").df().iloc[0]
print(f"Leakage test {CHECK_T}: {n_matched:,} of {n_stored:,} stored rows found in the data-cut rebuild; "
      f"feature values that differ: {int(bad.sum())}")
assert n_matched == n_stored and int(bad.sum()) == 0, 'features depend on data after t: stop and investigate'
os.makedirs('work/outputs', exist_ok=True)
json.dump({'snapshot_checked': CHECK_T, 'stored_rows': int(n_stored), 'rows_matched': int(n_matched),
           'feature_values_that_differ': int(bad.sum()), 'features_checked': len(PAST_COLS)},
          open('work/outputs/leakage_test.json', 'w'))

# ---- 2. The final ranked queue at the last snapshot (no label exists for it) ----------------------------
T_FINAL = SNAPSHOTS['final_list']
build_features_only(T_FINAL, 'feat_final')
final = con.sql(f"SELECT client_hash_id, n_elig, {', '.join(PAST_COLS)} FROM feat_final WHERE n_elig >= {MIN_CLIENT_PAGES_MODEL}").df()
for c in PAST_COLS:
    if pd.api.types.is_timedelta64_dtype(final[c]):
        final[c] = final[c].dt.days
final['is_spike'] = final['imp_last30'] >= 2 * np.maximum(final['imp_prev30'], final['imp_prev60'])

# Final model: same settings, trained on every labeled snapshot (training + validation + test)
train_final = pd.concat([train_all, val, test])
model = make_hgb(0).fit(make_X(train_final), train_final['y'].values)

pool = final[~final['is_spike']].copy()
pool['p_decline'] = model.predict_proba(make_X(pool))[:, 1]

# Descriptive state tags (past data only; thresholds fixed here, not validated against the label)
final['tag'] = 'stable'
final.loc[final['log_mom_rel'] >= 0.4, 'tag'] = 'growing'
final.loc[(final['log_mom_prev_rel'] <= -0.4) & (final['log_mom_rel'] >= 0.2), 'tag'] = 'recovering'
final.loc[final['is_spike'], 'tag'] = 'recent spike'
pool['tag'] = final.loc[pool.index, 'tag']

# Reason codes: watch-list conditions the page meets, ordered by how much the model leaned on them.
# They describe the page; they are not exact model attributions and not causes.
CV_HI = pool['cv_90'].quantile(0.90)
ACTIVE_LO = pool['active_days_90'].quantile(0.10)
conds = pd.DataFrame({
    'VOLATILE_DAILY_IMPRESSIONS': pool['cv_90'] >= CV_HI,
    'INTERMITTENT_VISIBILITY': pool['active_days_90'] <= ACTIVE_LO,
    'FALLING_VS_CLIENT': pool['log_mom_rel'] <= -0.4,
    'DEEP_POSITION': pool['pos_last30'] >= 30,
    'POSITION_SLIPPING': pool['pos_change'] >= 3,
    'NO_CLICKS_LAST_30D': pool['ctr_last30'] == 0,
})
pool['reason_codes'] = conds.apply(lambda r: ', '.join([k for k, v in r.items() if v][:3]), axis=1)

# Suggested review action (decision support, not a causal claim)
high_risk = pool['p_decline'] >= pool['p_decline'].quantile(0.95)
pool['action'] = 'monitor'
pool.loc[pool['tag'] == 'growing', 'action'] = 'protect'
pool.loc[high_risk, 'action'] = 'review: improve or monitor'
pool.loc[high_risk & (pool['imp_last30'] >= 1000), 'action'] = 'review now: improve or refresh'
pool.loc[high_risk & (pool['imp_last30'] < 1000) & conds['INTERMITTENT_VISIBILITY'] & conds['DEEP_POSITION'],
         'action'] = 'review for merge or prune'

client_rank = pool.groupby('client_hash_id').size().rank(ascending=False, method='first').astype(int)
pool['client_rank'] = pool['client_hash_id'].map(client_rank)
queue = pool.sort_values('p_decline', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

print(f"\nFinal list {T_FINAL}: {len(final):,} eligible pages from {final['client_hash_id'].nunique()} clients; "
      f"{int(final['is_spike'].sum()):,} tagged 'recent spike' (not scored); {len(pool):,} scored")
print(final['tag'].value_counts().to_string())
print(queue['action'].value_counts().to_string())

SHOW = ['rank', 'client_rank', 'p_decline', 'imp_last30', 'imp_prev30', 'tag', 'reason_codes', 'action']
print('\nTop 20 overall:')
print(queue[SHOW].head(20).round(3).to_string(index=False))
print('\nTop 20 among pages with >= 1,000 impressions in the last 30 days:')
hv = queue[queue['imp_last30'] >= 1000][SHOW].head(20).round(3)
print(hv.to_string(index=False))

queue[SHOW].head(50).round(4).to_csv('ranked_queue_sample.csv', index=False)   # no ids; review before committing

con.close()
gc.collect()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leakage test 2026-03-31: 64,669 of 64,669 stored rows found in the data-cut rebuild; feature values that differ: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Final list 2026-06-30: 77,251 eligible pages from 22 clients; 3,888 tagged 'recent spike' (not scored); 73,363 scored
tag
stable          57437
growing          8001
recovering       7925
recent spike     3888
action
monitor                           62603
protect                            7091
review: improve or monitor         2294
review for merge or prune           783
review now: improve or refresh      592

Top 20 overall:
 rank  client_rank  p_decline  imp_last30  imp_prev30     tag                                                            reason_codes                         action
    1            5      0.825       391.0       810.0  stable               INTERMITTENT_VISIBILITY, DEEP_POSITION, POSITION_SLIPPING      review for merge or prune
    2            5      0.813       161.0       407.0  stable  VOLATILE_DAILY_IMPRESSIONS, INTERMITTENT_VISIBILITY, FALLING_VS_CLIENT      review for merge or prune
    3            2      0.810       206.0       437.0  stable         

55

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
